# 08 - Documentation

**You will learn**: descriptions in YAML, doc blocks, the overview page, `dbt docs generate` / `dbt docs serve`, and the lineage graph.

**You will build**: the documentation site of your project.

dbt documentation lives **next to the code** and is generated from it, so it stays close to reality.
It contains descriptions, column lists with types, tests, the SQL of each model, and an interactive lineage graph.

In [ ]:
from helpers import *

## 1. Where does the documentation come from?

* `description:` on models, columns, sources and seeds in the YAML files;
* the **catalog** (the real columns and types, read from Databricks);
* the DAG (`ref` and `source`);
* the SQL code itself.

Let us measure how much of our project is documented today. This cell reads the files dbt generates in `target/`:

In [ ]:
dbt("docs generate")

In [ ]:
import json

def doc_coverage():
    manifest = json.loads((PROJECT / "target/manifest.json").read_text(encoding="utf-8"))
    catalog = json.loads((PROJECT / "target/catalog.json").read_text(encoding="utf-8"))
    rows = []
    for uid, node in catalog["nodes"].items():
        if not uid.startswith("model.alpsport."):
            continue
        declared = manifest["nodes"][uid].get("columns", {})
        cols = list(node["columns"])
        described = [c for c in cols if declared.get(c.lower(), declared.get(c, {})).get("description")]
        rows.append({"model": node["metadata"]["name"], "n_columns": len(cols), "described": len(described),
                     "model_description": bool(manifest["nodes"][uid].get("description"))})
    df = pd.DataFrame(rows).sort_values("model")
    df["coverage_%"] = (100 * df.described / df.n_columns).round(0)
    return df

cov = doc_coverage()
display(cov)
print(f"Overall column coverage: {100 * cov.described.sum() / cov.n_columns.sum():.0f}%")

Almost nothing is documented yet. Let us change that.

## 2. Describe a model and its columns

Descriptions go in the YAML next to the tests. **Exercise A.** In `models/gold/_gold.yml` add a `description` for `fct_sales` and for the columns
`order_line_id`, `net_amount` and `margin_amount`. For example:

```yaml
  - name: fct_sales
    description: >
      Sales fact table, one row per order line.
    columns:
      - name: net_amount
        description: Gross amount minus discount, in CHF.
```

(Keep the tests you already have under each column: a column entry can hold both `description` and `data_tests`.)

## 3. Doc blocks: write once, reuse everywhere

Some text is shared by several columns, for instance the meaning of `order_status` appears in silver and gold.
A **doc block** lives in a Markdown file and is referenced with `{{ doc('name') }}`.
The next cell writes `models/docs.md`, with three blocks:

* `order_status` and `loyalty_tier` for the columns,
* `__overview__`, a special block that becomes the **home page** of the site.

In [ ]:
%%writefile ../project/models/docs.md
{% docs __overview__ %}

# AlpSport analytics

Synthetic sales data of a Swiss sports retailer (11 stores + online shop), used in the dbt training.

| Layer | Catalog | Content |
|---|---|---|
| Bronze | `bronze.sports_shop` | raw tables generated by the notebook `generate_bronze_data` |
| Silver | `silver.sports_shop` | cleaned staging models (`stg_*`), the `countries` seed and the snapshots |
| Gold | `gold.sports_shop` | star schema (`dim_*`, `fct_sales`) and business marts (`mart_*`) |

{% enddocs %}

{% docs order_status %}

Life-cycle status of an order:

* `completed` - the order was paid and delivered
* `cancelled` - the order was cancelled before delivery
* `returned` - the customer sent the goods back after delivery (set by a later update in the source system)

{% enddocs %}

{% docs loyalty_tier %}

Loyalty programme level of the customer: `basic`, `silver`, `gold` or `platinum`.
Customers move up one tier from time to time.

{% enddocs %}


**Exercise B.** Use the block in a column description:

```yaml
      - name: order_status
        description: '{{ doc("order_status") }}'
```

(Quote the value: YAML would misread `{{`.)

## 4. The finished documentation

Describing every column of every model would take an afternoon. In practice you document what matters: **primary and foreign keys,
measures, and columns with a business meaning**. The finished YAML files of the project do exactly that.
Run the next cell to overwrite yours with them (`restore_checkpoint(8)`, it also writes `docs.md`), then regenerate the site
and measure the coverage again: it goes up to roughly a third of all columns, and every model has a description.

In [ ]:
restore_checkpoint(8)

In [ ]:
dbt("docs generate")

In [ ]:
cov = doc_coverage()
print(f"Overall column coverage: {100 * cov.described.sum() / cov.n_columns.sum():.0f}%")
cov

**Exercise C.** Pick the mart `mart_store_performance` and document **all** its columns in `models/gold/_gold.yml`
until its coverage is 100% (re-run `dbt docs generate` and the coverage cell to check).

## 5. Browse the site

Open a **terminal** in the project folder (with the venv activated) and run:

```bash
cd training/project
dbt docs serve --port 8080
```

Then open <http://localhost:8080>. (Stop the server with `Ctrl+C`.) Things to try:

1. The home page is the `__overview__` block.
2. In the left tree open **Projects > alpsport > models > gold > fct_sales**: description, columns, tests, and the compiled SQL at the bottom.
3. Click the **lineage graph** icon (bottom right). Use the selector `+fct_sales` to show only what feeds the fact table.
4. Find where `countries` (the seed) enters the graph.
5. Open a **source** (`bronze.sales_orders`) and read its description.

## Discussion

* Which descriptions are the most valuable for an analyst? (Business meaning, units, allowed values, grain.)
* Add documentation to your **pull request checklist**: new model = description on the model and its key columns.
* dbt can also check for it: the package `dbt_project_evaluator`, or `dbt run-operation` on the coverage.
* `meta:` and `exposures:` document owners and downstream dashboards.

## Recap

* Descriptions in YAML, shared text in doc blocks, `__overview__` for the home page.
* `dbt docs generate` builds `target/catalog.json` and `manifest.json`; `dbt docs serve` shows the site.
* The lineage graph answers "what breaks if I change this?".

---